In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
import lightgbm as lgb

In [2]:
df = pd.read_csv('dataset/feature_engineered_train.csv')
df.head()

,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,...,LargeVehicles_bin,road_score,road_capacity,lane_vehicle_interaction,geohash_mean_demand,weather_mean_demand,geohash_hour_mean,geohash_weather_mean,sin_day,cos_day
0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,16.382587,NaN,...,0,NaN,NaN,0,0.040048,NaN,0.034855,NaN,-0.781831,0.62349
1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,...,1,1.0,3.0,3,0.208766,0.094247,0.140998,0.211319,-0.781831,0.62349
2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,...,0,1.0,1.0,0,0.127931,0.094247,0.068046,0.135150,-0.781831,0.62349
3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,16.382587,Rainy,...,0,1.0,1.0,0,0.014381,0.094471,0.011243,0.011382,-0.781831,0.62349
4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,...,0,1.0,1.0,0,0.029300,0.094471,0.020746,0.021256,-0.781831,0.62349


In [3]:
target = "demand"
groups = df["geohash"]

In [4]:
def oof_target_encoding(df, col, target, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    global_mean = df[target].mean()

    encoded = np.zeros(len(df))

    for train_idx, val_idx in gkf.split(df, groups=df["geohash"]):
        train, val = df.iloc[train_idx], df.iloc[val_idx]

        means = train.groupby(col)[target].mean()
        encoded[val_idx] = df.iloc[val_idx][col].map(means).fillna(global_mean)

    return encoded

In [5]:
df["geohash_te"] = oof_target_encoding(df, "geohash", target)

In [6]:
global_mean = df[target].mean()
geohash_stats = df.groupby("geohash")["demand"].agg(["mean", "count"])

alpha = 10

geohash_te_map = (
    (geohash_stats["mean"] * geohash_stats["count"] + global_mean * alpha)
    / (geohash_stats["count"] + alpha)
).to_dict()

In [21]:
artifacts = {
    "global_mean": global_mean,
    "geohash_stats": geohash_stats,
    "geohash_te_map": geohash_te_map,
    "temp_median": df["Temperature"].median(),
    "geohash_mean": df.groupby("geohash")["demand"].mean(),
    "weather_mean": df.groupby("Weather")["demand"].mean(),
    "geohash_hour_mean": df.groupby(["geohash","hour"])["demand"].mean().to_dict(),
    "geohash_weather_mean": df.groupby(["geohash","Weather"])["demand"].mean().to_dict()
}
artifacts

{'global_mean': np.float64(0.09394238120172715),
 'geohash_stats':              mean  count
 geohash                 
 qp02yc   0.018498     12
 qp02yf   0.029433      1
 qp02yy   0.002902      2
 qp02yz   0.036564     30
 qp02z1   0.040048     33
 ...           ...    ...
 qp0dn4   0.007811      3
 qp0dn5   0.003714      7
 qp0dnh   0.001401      2
 qp0dnj   0.026414     55
 qp0dnn   0.004244      2
 
 [1249 rows x 2 columns],
 'geohash_te_map': {'qp02yc': 0.05279070670739287,
  'qp02yf': 0.08807785265819415,
  'qp02yy': 0.078768912789933,
  'qp02yz': 0.05090836483009065,
  'qp02z1': 0.05258130102828422,
  'qp02z3': 0.04113655254737355,
  'qp02z4': 0.06047796347565877,
  'qp02z5': 0.048606357786152105,
  'qp02z6': 0.050964196557133634,
  'qp02z7': 0.051408661717313925,
  'qp02z9': 0.045564354899506596,
  'qp02zc': 0.057355996767405396,
  'qp02zd': 0.06064684414585087,
  'qp02ze': 0.09131732194977638,
  'qp02zf': 0.0672928312052766,
  'qp02zg': 0.05299628190819405,
  'qp02zh': 0.129413

In [22]:
features = [
    # time
    "sin_time", "cos_time",
    "sin_day", "cos_day",
    "hour",

    # weather
    "Temperature",
    "temp_sq",
    "weather_temp",
    "temp_bin",

    # infrastructure
    "NumberofLanes",
    "road_capacity",
    "road_score",
    "Landmarks_bin",
    "LargeVehicles_bin",
    "lane_vehicle_interaction",

    # categorical
    "RoadType",
    "Weather",
    "demand_period",
    "time_bucket",

    # spatial
    "geohash_te",

    "geohash_mean_demand",
    "weather_mean_demand",
    "geohash_hour_mean",
    "geohash_weather_mean"
]

In [23]:
cat_cols = ["RoadType", "Weather", "temp_bin", "demand_period", "time_bucket", "weather_temp"]

for c in cat_cols:
    df[c] = df[c].astype("category")

In [24]:
params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.03,
    "num_leaves": 128,
    "max_depth": -1,
    "min_data_in_leaf": 30,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l2": 1.0,
    "verbosity": -1
}

In [25]:
x = df[features].copy()
y = df["demand"]

In [26]:
gkf = GroupKFold(n_splits=5)

oof_preds = np.zeros(len(df))

for fold, (train_idx, val_idx) in enumerate(gkf.split(x, y, groups=df["geohash"])):

    x_train, x_val = x.iloc[train_idx], x.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    train_data = lgb.Dataset(
        x_train,
        label=y_train,
        categorical_feature=cat_cols
    )

    val_data = lgb.Dataset(
        x_val,
        label=y_val,
        categorical_feature=cat_cols
    )

    model = lgb.train(
        params,
        train_data,
        valid_sets=[val_data],
        num_boost_round=5000,
        callbacks=[lgb.early_stopping(200)]
    )

    preds = model.predict(x_val)
    oof_preds[val_idx] = preds

    print("Fold R2:", r2_score(y_val, preds))

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[395]	valid_0's rmse: 0.0240698
Fold R2: 0.9708730575987792
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[254]	valid_0's rmse: 0.0227612
Fold R2: 0.9668317183990752
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[290]	valid_0's rmse: 0.0243531
Fold R2: 0.9674249353867411
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[327]	valid_0's rmse: 0.0229866
Fold R2: 0.9702518986756538
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[220]	valid_0's rmse: 0.0268541
Fold R2: 0.9754477225518415


In [27]:
print("OOF R2:", r2_score(y, oof_preds))

OOF R2: 0.9709167052428966


In [28]:
import joblib
joblib.dump(artifacts, "artifacts.pkl")
joblib.dump(model, "model.pkl")

['model.pkl']

In [29]:
test = pd.read_csv('dataset/test.csv', index_col='Index')
test.head()

,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
Index,,,,,,,,,
0,qp02z1,49,2:15,NaN,1,Not Allowed,No,NaN,NaN
1,qp02z9,49,2:15,Residential,1,Not Allowed,No,6.476213,Snowy
2,qp02yf,49,2:15,Residential,3,Allowed,Yes,22.318203,Sunny
3,qp02z6,49,2:15,Residential,2,Not Allowed,Yes,NaN,Rainy
4,qp02zd,49,2:15,Residential,1,Not Allowed,No,18.266162,Foggy


In [35]:
class DemandFeaturePipeline:
    
    def __init__(self, artifacts):
        """
        artifacts = dict containing all training-time statistics
        """
        self.global_mean = artifacts["global_mean"]
        self.geohash_te_map = artifacts["geohash_te_map"]
        self.temp_median = artifacts["temp_median"]
        self.geohash_mean = artifacts["geohash_mean"]
        self.weather_mean = artifacts["weather_mean"]
        self.geohash_hour_mean = artifacts["geohash_hour_mean"]
        self.geohash_weather_mean = artifacts["geohash_weather_mean"]
        
        self.road_map = {
            'Highway':3,
            'Street':2,
            'Residential':1
        }

        self.cat_cols = [
            "RoadType",
            "Weather",
            "temp_bin",
            "demand_period",
            "time_bucket",
            "weather_temp"
        ]

    def transform(self, df):

        df = df.copy()

        # ---------------------------
        # 1. Clean categorical cols
        # ---------------------------
        cat_cols_raw = [
            'geohash','RoadType','LargeVehicles',
            'Landmarks','Weather'
        ]

        for c in cat_cols_raw:
            df[c] = df[c].astype(str).str.strip()

        # ---------------------------
        # 2. Missing flags
        # ---------------------------
        df['temp_missing'] = df['Temperature'].isna().astype(int)
        df['weather_missing'] = df['Weather'].isna().astype(int)
        df['roadtype_missing'] = df['RoadType'].isna().astype(int)

        # ---------------------------
        # 3. Fill temperature
        # ---------------------------
        df['Temperature'] = df['Temperature'].fillna(self.temp_median)

        # ---------------------------
        # 4. Time parsing
        # ---------------------------
        df[['hour','minute']] = df['timestamp'].apply(
            lambda x: pd.Series(list(map(int, x.split(':'))))
        )

        df['total_minutes'] = df['hour'] * 60 + df['minute']

        df['sin_time'] = np.sin(2*np.pi*df['total_minutes']/1440)
        df['cos_time'] = np.cos(2*np.pi*df['total_minutes']/1440)

        # ---------------------------
        # 5. Time buckets
        # ---------------------------
        def get_time_bucket(h):
            if h < 6: return 'Night'
            elif h < 12: return 'Morning'
            elif h < 18: return 'Afternoon'
            return 'Evening'

        df['time_bucket'] = df['hour'].apply(get_time_bucket)

        def demand_period(h):
            if h <= 4: return "Night"
            elif h <= 9: return "MorningRise"
            elif h <= 14: return "Peak"
            elif h <= 17: return "Decline"
            elif h <= 20: return "LowDemand"
            return "Recovery"

        df['demand_period'] = df['hour'].apply(demand_period)

        # ---------------------------
        # 6. Temperature bins
        # ---------------------------
        df['temp_bin'] = pd.cut(
            df['Temperature'],
            bins=[-np.inf,10,20,30,40,np.inf],
            labels=['VeryCold','Cold','Moderate','Warm','Hot']
        )

        df['temp_sq'] = df['Temperature'] ** 2

        # ---------------------------
        # 7. Weather interaction
        # ---------------------------
        df['weather_temp'] = (
            df['Weather'].astype(str) + '_' + df['temp_bin'].astype(str)
        )

        # ---------------------------
        # 8. Binary encodings
        # ---------------------------
        df['Landmarks_bin'] = df['Landmarks'].map({'Yes':1,'No':0})
        df['LargeVehicles_bin'] = df['LargeVehicles'].map({'Allowed':1,'Not Allowed':0})

        # ---------------------------
        # 9. Road features
        # ---------------------------
        df['road_score'] = df['RoadType'].map(self.road_map)
        df['road_capacity'] = df['road_score'] * df['NumberofLanes']
        df['lane_vehicle_interaction'] = df['NumberofLanes'] * df['LargeVehicles_bin']

        # ---------------------------
        # 10. Cyclic day
        # ---------------------------
        df['sin_day'] = np.sin(2*np.pi*df['day']/7)
        df['cos_day'] = np.cos(2*np.pi*df['day']/7)

        # ---------------------------
        # 11. SAFE target encodings (NO RECOMPUTE)
        # ---------------------------
        df['geohash_mean_demand'] = df['geohash'].map(self.geohash_mean)
        df['weather_mean_demand'] = df['Weather'].map(self.weather_mean)

        df['geohash_hour_mean'] = list(
            map(
                lambda x: self.geohash_hour_mean.get(x, self.geohash_mean.get(x[0], 0)),
                zip(df['geohash'], df['hour'])
            )
        )

        df['geohash_weather_mean'] = list(
            map(
                lambda x: self.geohash_weather_mean.get(x, self.geohash_mean.get(x[0], 0)),
                zip(df['geohash'], df['Weather'])
            )
        )

        df["geohash_te"] = df["geohash"].map(self.geohash_te_map)
        df["geohash_te"] = df["geohash_te"].fillna(self.global_mean)

        # ---------------------------
        # 13. Final type casting
        # ---------------------------
        for c in self.cat_cols:
            df[c] = df[c].astype('category')

        return df

In [36]:
class DemandModel:
    
    def __init__(self, model, pipeline, features):
        self.model = model
        self.pipeline = pipeline
        self.features = features

    def predict(self, df):

        df = self.pipeline.transform(df)
        X = df[self.features]

        preds = self.model.predict(X)

        return preds

In [37]:
pipeline = DemandFeaturePipeline(artifacts)

In [38]:
model_wrapper = DemandModel(
    model=model,
    pipeline=pipeline,
    features=features
)

In [39]:
preds = model_wrapper.predict(test)

In [40]:
preds

array([0.07091149, 0.01196211, 0.03442036, ..., 0.01103827, 0.1244087 ,
       0.02170877], shape=(41778,))

In [41]:
test["preds"] = preds
test.head()

,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,preds
Index,,,,,,,,,,
0,qp02z1,49,2:15,NaN,1,Not Allowed,No,NaN,NaN,0.070911
1,qp02z9,49,2:15,Residential,1,Not Allowed,No,6.476213,Snowy,0.011962
2,qp02yf,49,2:15,Residential,3,Allowed,Yes,22.318203,Sunny,0.034420
3,qp02z6,49,2:15,Residential,2,Not Allowed,Yes,NaN,Rainy,0.079664
4,qp02zd,49,2:15,Residential,1,Not Allowed,No,18.266162,Foggy,0.068133


In [43]:
test.to_csv("dataset/submission.csv", columns=["preds"], index_label="Index")